# Phân tích chuỗi thời gian và dự báo điện năng tiêu thụ hằng ngày

Notebook duy nhất thực hiện toàn bộ quy trình từ CSV gốc đến dữ liệu ngày, EDA, huấn luyện, đánh giá và dự báo. Chạy các ô theo thứ tự từ trên xuống bằng **Restart Kernel → Run All**. Các hàm được định nghĩa ngay trong notebook để đọc và kiểm tra.

**Giả định:** `Load Demand (kW)` là công suất trung bình trong khoảng 15 phút; điện năng ngày bằng tổng công suất × 0,25 giờ. Giả định và nguồn gốc dữ liệu thực hay mô phỏng chưa được xác minh.

**Nội dung:** 1. Thiết lập; 2. Kiểm tra và tổng hợp dữ liệu; 3. EDA; 4. Đặc trưng và giao thức; 5. Thiết kế mô hình; 6. Huấn luyện và lựa chọn; 7. Đánh giá test; 8. Dự báo 7 ngày; 9. Kết luận.

## 1. Thiết lập môi trường

### 1.1. Nạp thư viện

Nạp thư viện xử lý bảng, biểu đồ, học máy và chuỗi thời gian. Cài requirements.txt trước khi chạy notebook.

In [ ]:
from pathlib import Path
import json, time, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace.exponential_smoothing import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from IPython.display import display


ModuleNotFoundError: No module named 'numpy'

### 1.2. Xác định đầu vào và thư mục đầu ra

Tìm thư mục chứa CSV gốc từ vị trí mở notebook. Tạo thư mục dữ liệu ngày và giữ cấu trúc outputs để lưu biểu đồ, bảng, dự báo và mô hình.

In [ ]:
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'load_forecasting_dataset_corrected.csv').exists()))
OUT = ROOT / 'outputs'
(ROOT / 'data/processed').mkdir(parents=True, exist_ok=True)
for folder in ['figures', 'tables', 'predictions', 'models']:
    (OUT / folder).mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)
print('Thư mục dự án:', ROOT)


### 1.3. Tiện ích lưu biểu đồ

Hàm dưới đây lưu hình PNG vào outputs/figures, hiển thị trong notebook rồi đóng hình.

In [ ]:
def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT / 'figures' / f'{name}.png', dpi=150)
    plt.show()
    plt.close()


## 2. Xác minh dữ liệu và tạo chuỗi điện năng ngày

### 2.1. Đọc dữ liệu và chuyển timestamp

Đọc CSV gốc, chuyển thời gian theo tháng/ngày/năm và hiển thị mẫu để kiểm tra.

In [ ]:
df = pd.read_csv(ROOT / 'load_forecasting_dataset_corrected.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%m/%d/%Y %H:%M')
display(df.head())
print(df.shape)


### 2.2. Kiểm tra tính đầy đủ và hợp lệ

Thống kê ô thiếu, timestamp trùng, thứ tự thời gian, công suất không hợp lệ và khoảng đo khác 15 phút. Đối chiếu các cột lịch với timestamp. Lưu báo cáo JSON; dừng nếu lỗi công suất hoặc trục thời gian để điều tra trước khi tổng hợp.

In [ ]:
quality = {'rows': len(df), 'missing_cells': int(df.isna().sum().sum()), 'duplicate_times': int(df.Timestamp.duplicated().sum()), 'sorted': bool(df.Timestamp.is_monotonic_increasing)}
df = df.sort_values('Timestamp').set_index('Timestamp')
p = df['Load Demand (kW)']
quality['invalid_load'] = int((~np.isfinite(p) | (p < 0)).sum())
quality['irregular_intervals'] = int((df.index.to_series().diff().dropna() != pd.Timedelta(minutes=15)).sum())
for col, expected in [('Day of Week', df.index.dayofweek), ('Hour of Day', df.index.hour), ('Month', df.index.month)]:
    quality[col + '_mismatch'] = int((df[col].to_numpy() != expected).sum())
(OUT / 'tables' / 'data_quality.json').write_text(json.dumps(quality, indent=2))
assert quality['duplicate_times'] == quality['invalid_load'] == quality['irregular_intervals'] == 0, quality
display(quality)


### 2.3. Tổng hợp và lưu daily_energy.csv

Tính điện năng ngày bằng tổng 96 giá trị kW × 0,25 giờ; yêu cầu mỗi ngày đủ bản ghi. Lấy trung bình nhiệt độ, độ ẩm và gió. Không cộng mưa khi chưa rõ định nghĩa. Ô này trực tiếp tạo data/processed/daily_energy.csv.

In [ ]:
daily = pd.DataFrame({'energy_kwh': p.resample('D').sum(min_count=96) * 0.25, 'record_count': p.resample('D').count()})
assert daily.record_count.eq(96).all() and daily.energy_kwh.notna().all()
for col in ['Temperature (°C)', 'Humidity (%)', 'Wind Speed (m/s)']:
    daily[col] = df[col].resample('D').mean()
daily.index.name = 'date'
daily.to_csv(ROOT / 'data/processed/daily_energy.csv')
daily = daily.asfreq('D')
display(daily.head())
display(daily.describe())


## 3. Phân tích khám phá chuỗi thời gian

Chỉ phân tích đến hết 2024 để tránh dùng test khi lựa chọn mô hình. Tương quan không chứng minh quan hệ nhân quả.

### 3.1. Xu hướng và trung bình trượt

Vẽ điện năng ngày cùng trung bình trượt 7 và 30 ngày.

In [ ]:
d = daily.loc[:'2024-12-31']
eda_y = d.energy_kwh
pd.DataFrame({'Ngày': eda_y, 'TB 7 ngày': eda_y.rolling(7).mean(), 'TB 30 ngày': eda_y.rolling(30).mean()}).plot(figsize=(13, 4))
plt.ylabel('kWh/ngày')
savefig('trend')


### 3.2. Phân phối theo lịch

Vẽ histogram và boxplot theo thứ, tháng để khảo sát phân bố.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(eda_y, bins=35)
ax[1].boxplot([eda_y[eda_y.index.dayofweek == i] for i in range(7)])
ax[1].set_title('Thứ: 1 = thứ Hai')
ax[2].boxplot([eda_y[eda_y.index.month == i] for i in range(1, 13)])
ax[2].set_title('Tháng')
savefig('distributions')


### 3.3. Heatmap năm–tháng

So sánh điện năng trung bình giữa các tháng và năm.

In [ ]:
matrix = pd.DataFrame({'year': eda_y.index.year, 'month': eda_y.index.month, 'energy': eda_y.values}).pivot_table(index='year', columns='month', values='energy')
plt.figure(figsize=(10, 4))
plt.imshow(matrix, aspect='auto')
plt.yticks(range(len(matrix)), matrix.index)
plt.xticks(range(12), range(1, 13))
plt.colorbar(label='kWh/ngày')
savefig('year_month')


### 3.4. Phân rã STL

Tách xu hướng, mùa vụ tuần và phần dư với chu kỳ 7 ngày.

In [ ]:
STL(eda_y, period=7, robust=True).fit().plot()
savefig('stl')


### 3.5. Tự tương quan

Vẽ ACF/PACF đến 60 độ trễ để khảo sát phụ thuộc thời gian.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(eda_y, lags=60, ax=ax[0])
plot_pacf(eda_y, lags=60, ax=ax[1], method='ywm')
savefig('acf_pacf')


### 3.6. Quan hệ với thời tiết

Vẽ quan hệ với nhiệt độ, độ ẩm và gió; chỉ mang ý nghĩa mô tả.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, c in zip(ax, d.columns[2:]):
    a.scatter(d[c], eda_y, s=4, alpha=0.4)
    a.set_xlabel(c)
    a.set_ylabel('kWh/ngày')
savefig('weather')


### 3.7. Tóm tắt định lượng

Hiển thị tương quan và chênh lệch trung bình giữa các thứ. Kết hợp STL/ACF để diễn giải tính ổn định của chu kỳ.

In [ ]:
display(d.corr(numeric_only=True))
weekly = d.groupby(d.index.dayofweek).energy_kwh.mean()
print(f'Chênh lệch giữa các thứ: {weekly.max() - weekly.min():,.2f} kWh/ngày')


## 4. Thiết kế đặc trưng và giao thức đánh giá

### 4.1. Chia tập theo thời gian

Train 2020–2023; validation 2024; test 01–05/2025. Không chia ngẫu nhiên. Validation chọn mô hình; test chỉ đánh giá sau khi khóa cấu hình.

In [ ]:
y = daily.energy_kwh
assert y.notna().all()
train = y.loc[:'2023-12-31']
val = y.loc['2024']
test = y.loc['2025-01-01':]
display(pd.DataFrame([{'split': name, 'start': s.index.min(), 'end': s.index.max(), 'days': len(s)} for name, s in [('train', train), ('validation', val), ('test', test)]]))


### 4.2. Tạo đặc trưng lịch sử và lịch

Định nghĩa lag 1, 2, 7, 14, 28 ngày; thống kê trượt dịch một ngày; thứ, tháng, cuối tuần và chu kỳ năm. Không dùng thời tiết thực tế tương lai.

In [ ]:
def features(y):
    x = pd.DataFrame(index=y.index)
    for lag in [1, 2, 7, 14, 28]:
        x[f'lag_{lag}'] = y.shift(lag)
    for w in [7, 14, 28]:
        x[f'mean_{w}'] = y.shift(1).rolling(w).mean()
        x[f'std_{w}'] = y.shift(1).rolling(w).std()
    x['weekday'] = x.index.dayofweek
    x['month'] = x.index.month
    x['weekend'] = (x.index.dayofweek >= 5).astype(int)
    x['annual_sin'] = np.sin(2 * np.pi * x.index.dayofyear / 365.25)
    x['annual_cos'] = np.cos(2 * np.pi * x.index.dayofyear / 365.25)
    return x


### 4.3. Kiểm chứng không sử dụng tương lai

Thay đổi điện năng từ một ngày trở đi và kiểm tra đặc trưng đến ngày đó vẫn giữ nguyên. Hiển thị bảng đặc trưng; các dòng đầu thiếu lag được loại khi huấn luyện.

In [ ]:
x = features(y)
cut = pd.Timestamp('2024-06-01')
changed = y.copy()
changed.loc[cut:] += 100000
pd.testing.assert_frame_equal(x.loc[:cut], features(changed).loc[:cut])
assert x.loc[cut, 'lag_1'] == y.loc[cut - pd.Timedelta(days=1)]
display(x.dropna().head())
print('Đạt kiểm tra tính nhân quả của đặc trưng.')


## 5. Thiết kế mô hình và phương pháp dự báo

### 5.1. Các cấu hình thử nghiệm

So sánh Naive, Seasonal Naive, ETS mùa vụ cộng chu kỳ 7, ba cấu hình SARIMA, Random Forest và HistGradientBoosting. Giữ nguyên phạm vi tìm kiếm của dự án.

In [ ]:
CONFIGS = [{'name': 'Naive', 'kind': 'naive'}, {'name': 'SeasonalNaive', 'kind': 'seasonal'}, {'name': 'ETS_level', 'kind': 'ets', 'trend': False}, {'name': 'ETS_trend', 'kind': 'ets', 'trend': True}, *[{'name': f'SARIMA_{p}{d}{q}', 'kind': 'sarima', 'order': [p, d, q]} for p, d, q in [(1, 0, 0), (1, 0, 1), (1, 1, 1)]], *[{'name': f'RF_leaf{leaf}', 'kind': 'rf', 'leaf': leaf} for leaf in [2, 7]], *[{'name': f'HGB_leaf{leaf}', 'kind': 'hgb', 'leaf': leaf} for leaf in [7, 15]]]
display(pd.DataFrame(CONFIGS))


### 5.2. Khởi tạo và huấn luyện

Hàm chọn mô hình theo cấu hình. ETS tối đa 150 vòng lặp, SARIMA 100; ML cố định random_state=42 và loại dòng thiếu lag.

In [ ]:
def fit_model(y, c):
    if c['kind'] in ['naive', 'seasonal']:
        return None
    if c['kind'] == 'ets':
        return ExponentialSmoothing(y, trend=c['trend'], seasonal=7, initialization_method='estimated').fit(disp=False, maxiter=150)
    if c['kind'] == 'sarima':
        return SARIMAX(y, order=tuple(c['order']), seasonal_order=(1, 0, 0, 7), trend='c', enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100)
    model = RandomForestRegressor(n_estimators=150, min_samples_leaf=c['leaf'], random_state=42, n_jobs=-1) if c['kind'] == 'rf' else HistGradientBoostingRegressor(max_iter=150, max_leaf_nodes=c['leaf'], early_stopping=False, random_state=42)
    x = features(y).dropna()
    return model.fit(x, y.loc[x.index])


### 5.3. Dự báo từ lịch sử có sẵn

Baseline dùng ngày trước hoặc tuần trước. Thống kê dự báo trực tiếp nhiều bước; ML dự báo đệ quy bằng chính giá trị dự báo khi chưa có thực tế.

In [ ]:
def forecast(model, history, dates, c):
    if c['kind'] in ['ets', 'sarima']:
        return np.asarray(model.forecast(len(dates)))
    h = history.copy()
    result = []
    for date in dates:
        if c['kind'] == 'naive':
            pred = h.iloc[-1]
        elif c['kind'] == 'seasonal':
            pred = h.iloc[-7]
        else:
            row = features(pd.concat([h, pd.Series([np.nan], index=[date])])).iloc[[-1]]
            pred = model.predict(row)[0]
        result.append(float(pred))
        h.loc[date] = pred
    return np.array(result)


### 5.4. Đánh giá cuốn chiếu

Mỗi lần chỉ dùng lịch sử đã biết. Sau mỗi khối dự báo, thêm thực tế vào lịch sử; ETS/SARIMA cập nhật trạng thái, giữ tham số cố định. ML giữ mô hình và cập nhật đặc trưng. Khối cuối không đủ chân trời được bỏ.

In [ ]:
def backtest(train, test, c, horizon=1):
    start = time.perf_counter()
    model = fit_model(train, c)
    fit_seconds = time.perf_counter() - start
    history = train.copy()
    rows = []
    start = time.perf_counter()
    for offset in range(0, len(test) - horizon + 1, horizon):
        block = test.iloc[offset:offset + horizon]
        pred = forecast(model, history, block.index, c)
        for step, (date, actual, value) in enumerate(zip(block.index, block.values, pred), 1):
            rows.append({'date': date, 'actual': actual, 'prediction': value, 'model': c['name'], 'horizon': step, 'origin': history.index[-1]})
        history = pd.concat([history, block])
        if c['kind'] in ['ets', 'sarima']:
            model = model.extend(block)
    return (pd.DataFrame(rows), fit_seconds, time.perf_counter() - start)


### 5.5. Tính MAE và RMSE

Kiểm tra giá trị hữu hạn rồi tính sai số, đơn vị kWh/ngày. MAE đo sai lệch tuyệt đối trung bình; RMSE nhạy với sai số lớn.

In [ ]:
def metrics(p):
    assert len(p) and np.isfinite(p[['actual', 'prediction']]).all().all()
    return {'MAE': mean_absolute_error(p.actual, p.prediction), 'RMSE': np.sqrt(mean_squared_error(p.actual, p.prediction))}


## 6. Huấn luyện và lựa chọn trên validation

### 6.1. Chạy các cấu hình

Huấn luyện trên train và dự báo cuốn chiếu validation. Ghi thời gian, lỗi và cảnh báo; loại cấu hình có cảnh báo hội tụ khỏi lựa chọn.

In [ ]:
rows = []
failures = []
candidates = []
for c in CONFIGS:
    print('Validation:', c['name'], flush=True)
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            p, ft, pt = backtest(train, val, c)
        converged = not any(('converg' in str(w.message).lower() for w in caught))
        rows.append({'model': c['name'], 'kind': c['kind'], **metrics(p), 'fit_seconds': ft, 'prediction_seconds': pt, 'converged': converged, 'warnings': ' | '.join(sorted(set((str(w.message) for w in caught))))})
        if converged:
            candidates.append(c)
    except Exception as e:
        failures.append({'model': c['name'], 'error': repr(e)})


### 6.2. So sánh và khóa lựa chọn

Lưu bảng validation và lỗi. Chọn MAE thấp nhất, RMSE phá hòa; chọn cấu hình tốt nhất mỗi nhóm để so sánh test. Lưu cấu hình chọn trước khi chạy test.

In [ ]:
table = pd.DataFrame(rows).sort_values(['MAE', 'RMSE'])
table.to_csv(OUT / 'tables/validation.csv', index=False)
(OUT / 'tables/failures.json').write_text(json.dumps(failures, indent=2))
valid = table[table.converged]
best_name = valid.iloc[0].model
chosen = next((c for c in candidates if c['name'] == best_name))
selected = [next((c for c in candidates if c['name'] == g.iloc[0].model)) for _, g in valid.groupby('kind', sort=False)]
(OUT / 'models/selected_config.json').write_text(json.dumps(chosen, indent=2))
display(table)
print('Cấu hình được chọn:', chosen)


## 7. Đánh giá trên test và phân tích sai số

### 7.1. Huấn luyện lại và dự báo test

Huấn luyện cấu hình đã chọn của mỗi nhóm bằng train + validation. Đánh giá trên cùng các ngày test, ghi thời gian và dự báo.

In [ ]:
results = []
predictions = []
for c in selected:
    print('Test:', c['name'], flush=True)
    p, ft, pt = backtest(pd.concat([train, val]), test, c)
    results.append({'model': c['name'], **metrics(p), 'fit_seconds': ft, 'prediction_seconds': pt})
    predictions.append(p)


### 7.2. Xuất bảng sai số và dự báo

Tính MAE/RMSE, mức cải thiện so với Seasonal Naive và lưu CSV. Cải thiện âm nghĩa là kém baseline.

In [ ]:
summary = pd.DataFrame(results)
baseline = summary.loc[summary.model == 'SeasonalNaive', 'MAE'].iloc[0]
summary['MAE_improvement_percent'] = 100 * (baseline - summary.MAE) / baseline
summary.to_csv(OUT / 'tables/test_metrics.csv', index=False)
allp = pd.concat(predictions)
allp.to_csv(OUT / 'predictions/test.csv', index=False)
display(summary)


### 7.3. So sánh thực tế và dự báo

Vẽ toàn tập test và 60 ngày cuối, lưu PNG để dùng trong báo cáo.

In [ ]:
wide = allp.pivot(index='date', columns='model', values='prediction')
wide['Actual'] = test
for name, frame in [('test_full', wide), ('test_last60', wide.tail(60))]:
    frame.plot(figsize=(14, 5))
    plt.ylabel('kWh/ngày')
    savefig(name)


### 7.4. Phân tích phần dư

Với mô hình đã chọn từ validation, vẽ phân phối phần dư, ACF phần dư và MAE theo thứ.

In [ ]:
p = allp[allp.model == best_name].copy()
p['residual'] = p.actual - p.prediction
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(p.residual, bins=25)
plot_acf(p.residual, lags=30, ax=ax[1])
p.groupby(p.date.dt.dayofweek).residual.apply(lambda x: x.abs().mean()).plot.bar(ax=ax[2])
savefig('residuals')


## 8. Đánh giá nhiều bước và dự báo 7 ngày tiếp theo

### 8.1. Backtest chân trời 7 ngày

Dùng cấu hình đã khóa và Seasonal Naive. Dự báo các khối 7 ngày không chồng lấn trên validation và test; không dùng thực tế trong khối khi dự báo. Lưu sai số riêng từng chân trời.

In [ ]:
c = json.loads((OUT / 'models/selected_config.json').read_text())
scores = []
for label, train, test in [('validation', y.loc[:'2023-12-31'], y.loc['2024']), ('test', y.loc[:'2024-12-31'], y.loc['2025-01-01':])]:
    for config in [c, {'name': 'SeasonalNaive', 'kind': 'seasonal'}]:
        p, _, _ = backtest(train, test, config, 7)
        p.to_csv(OUT / f"predictions/{label}_7day_{config['name']}.csv", index=False)
        for h, g in p.groupby('horizon'):
            scores.append({'split': label, 'model': config['name'], 'horizon': h, **metrics(g)})
scores = pd.DataFrame(scores)
scores.to_csv(OUT / 'tables/multistep.csv', index=False)
display(scores)


### 8.2. Biểu đồ sai số theo chân trời

So sánh MAE ngày 1–7 trên test. Kết quả nhiều bước được đánh giá riêng với dự báo một ngày.

In [ ]:
for name, group in scores[scores.split == 'test'].groupby('model'):
    plt.plot(group.horizon, group.MAE, marker='o', label=name)
plt.xlabel('Chân trời (ngày)')
plt.ylabel('MAE (kWh/ngày)')
plt.legend()
savefig('horizon_error')


### 8.3. Huấn luyện cuối và dự báo tương lai

Huấn luyện bằng toàn bộ lịch sử, dự báo 7 ngày sau ngày cuối dữ liệu (01–07/06/2025). Lưu CSV và pickle gồm mô hình, cấu hình, lịch sử. Không tạo khoảng dự báo khi chưa kiểm tra độ bao phủ.

In [ ]:
model = fit_model(y, c)
dates = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=7)
pred = pd.Series(forecast(model, y, dates, c), index=dates, name='forecast_kwh')
pred.to_csv(OUT / 'predictions/future_7days.csv', index_label='date')
with open(OUT / 'models/final_model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'config': c, 'history': y}, f)
y.tail(60).plot(figsize=(12, 4), label='Lịch sử')
pred.plot(marker='o', label='Dự báo')
plt.legend()
plt.ylabel('kWh/ngày')
savefig('future_7days')
display(pred.to_frame())


## 9. Kết luận và khả năng tái lập

### 9.1. Kết luận từ kết quả thực tế

Tạo kết luận bằng số đo test của mô hình đã chọn từ validation, lưu văn bản. Không điều chỉnh mô hình theo test.

In [ ]:
row = summary.set_index('model').loc[chosen['name']]
message = f"Mô hình chọn từ validation: {chosen['name']}. Test MAE={row.MAE:,.2f}; RMSE={row.RMSE:,.2f} kWh/ngày. Cải thiện MAE so với Seasonal Naive: {row.MAE_improvement_percent:.2f}%."
print(message)
print('Cải thiện âm nghĩa là mô hình không vượt baseline trên test.')
(OUT / 'tables/conclusion.txt').write_text(message, encoding='utf-8')


### 9.2. Lưu phiên bản môi trường

Ghi phiên bản Python và các thư viện chính để tái lập lần chạy.

In [ ]:
import platform
from importlib.metadata import version
environment = {'python': platform.python_version(), **{name: version(name) for name in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsmodels']}}
(OUT / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
display(environment)


### 9.3. Giới hạn diễn giải

Nguồn gốc dữ liệu và ý nghĩa công suất trung bình 15 phút chưa được xác minh. Tập test gồm 5 tháng nên chưa đại diện đầy đủ cả năm. Tương quan thời tiết chỉ mô tả. Chất lượng dự báo 7 ngày phải dựa vào backtest nhiều bước; chưa có khoảng dự báo được hiệu chỉnh. Nếu mô hình không vượt baseline, giữ nguyên kết quả và báo cáo trung thực.